# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ 5c87dcd7-72a4-4b40-a28a-23d2db… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ 2bba5b6a-295f-4bc2-85fe-685c5a… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [4]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 8,
 'numFilesRemoved': 8,
 'filesAdded': '{"avg":2768.75,"max":2770,"min":2766,"totalFiles":8,"totalSize":22150}',
 'filesRemoved': '{"avg":2687.875,"max":2689,"min":2687,"totalFiles":8,"totalSize":21503}',
 'partitionsOptimized': 8,
 'numBatches': 8,
 'totalConsideredFiles': 8,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [5]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)

shape: (24, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-08 ┆ claude-hai ┆ 574.0      ┆ 1118.3    ┆ 4855783   ┆ 2403003   ┆ 0.042171  ┆ 13.496638 │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

### Bằng chứng bổ sung: kiểm tra đầy đủ điều kiện Gold

In [6]:
for _name, _p in [('bronze', BRONZE), ('silver', SILVER), ('gold', GOLD)]:
    _dt = DeltaTable(_p)
    print(f'{_name:<7} v{_dt.version()}  rows={_dt.to_pyarrow_table().num_rows:>7,}  {_p}')

_g = gold_df.sort(['date', 'model'])
_cells = _g.group_by('date').agg(pl.col('model').n_unique().alias('n_models'))
gold_checks = {
    'Silver < Bronze': silver_n < bronze_n,
    '>= 7 dates': n_dates >= 7,
    '3 models on every date': _cells['n_models'].min() == 3,
    'p50 <= p95 on every row': (_g['p50_latency_ms'] <= _g['p95_latency_ms']).all(),
    'cost_usd > 0 on every row': (_g['cost_usd'] > 0).all(),
    'error_rate in [0, 1]': _g['error_rate'].is_between(0, 1).all(),
}
for k, v in gold_checks.items():
    print(f'  [{"PASS" if v else "FAIL"}] {k}')
assert all(gold_checks.values())

with pl.Config(tbl_rows=30, tbl_cols=10, tbl_width_chars=140, fmt_str_lengths=20):
    print(_g.select('date', 'model', 'p50_latency_ms', 'p95_latency_ms',
                    pl.col('error_rate').round(4), pl.col('cost_usd').round(2)))


bronze  v0  rows=200,000  D:\vinuni AI\lab18\K4-Track02-Day18-Lakehouse-Lab\_lakehouse\bronze\llm_calls_raw


silver  v0  rows=190,052  D:\vinuni AI\lab18\K4-Track02-Day18-Lakehouse-Lab\_lakehouse\silver\llm_calls
gold    v1  rows=     24  D:\vinuni AI\lab18\K4-Track02-Day18-Lakehouse-Lab\_lakehouse\gold\llm_daily_metrics
  [PASS] Silver < Bronze
  [PASS] >= 7 dates
  [PASS] 3 models on every date
  [PASS] p50 <= p95 on every row
  [PASS] cost_usd > 0 on every row
  [PASS] error_rate in [0, 1]
shape: (24, 6)
┌────────────┬───────────────────┬────────────────┬────────────────┬────────────┬──────────┐
│ date       ┆ model             ┆ p50_latency_ms ┆ p95_latency_ms ┆ error_rate ┆ cost_usd │
│ ---        ┆ ---               ┆ ---            ┆ ---            ┆ ---        ┆ ---      │
│ date       ┆ str               ┆ f64            ┆ f64            ┆ f64        ┆ f64      │
╞════════════╪═══════════════════╪════════════════╪════════════════╪════════════╪══════════╡
│ 2026-04-01 ┆ claude-haiku-4-5  ┆ 567.0          ┆ 1121.0         ┆ 0.0494     ┆ 33.18    │
│ 2026-04-01 ┆ claude-opus-4-7   ┆ 304

<!-- explanation -->
## Giải thích kết quả — NB4

**Ba bảng trên storage** (`_lakehouse/`): `bronze/llm_calls_raw` (200 000 dòng), `silver/llm_calls`
(190 052 dòng, partition theo `date`), `gold/llm_daily_metrics` (24 dòng, partition theo `date`, Z-order theo `model`).

**Silver < Bronze:** dedup theo `request_id` (`ROW_NUMBER() ... rn = 1`) loại **9 948** dòng — đúng bằng số
retry/duplicate mà generator cố ý chèn (190 052 request_id duy nhất). Đây là vấn đề "at-least-once delivery":
client/gateway retry khiến cùng một request được log nhiều lần; nếu không dedup thì đếm request, token và
chi phí đều bị thổi phồng ~5%. Silver cũng ép kiểu và loại JSON lỗi (`model IS NOT NULL`).

**Gold:** cell bằng chứng bổ sung kiểm tra và PASS tất cả: 8 ngày (2026-04-01 → 04-08) × 3 model trên **mọi** ngày,
p50 ≤ p95 ở mọi dòng (ví dụ opus ~3 000 / ~6 000 ms, haiku ~565 / ~1 130 ms), `cost_usd > 0`,
`error_rate` ∈ [0, 1] (~4–6%). Ngày 04-08 có chi phí thấp hơn rõ (haiku 13.5 vs ~46 USD) vì là ngày cuối chỉ có
một phần dữ liệu — cửa sổ 7×24h của generator lệch sang ngày thứ 8; dashboard nên đánh dấu ngày chưa đủ.

**Cách tính có phù hợp không?** `error_rate = AVG(status <> 'ok')` đúng với dữ liệu đầu vào vì mọi dòng Silver đều
có `status`. `cost_usd = Σprompt × giá_in/1M + Σcompletion × giá_out/1M` dùng bảng giá minh họa của lab (không phải
giá thật). Dashboard đọc Gold vì Gold chỉ 24 dòng đã tổng hợp sẵn — quét nhỏ, nhanh, và mọi dashboard dùng chung
một định nghĩa metric thay vì mỗi người tự viết SQL trên 190K dòng Silver.